In [15]:
from WindPy import w
import numpy as np
import pandas as pd
import datetime
import os
import json
from tqdm import tqdm

w.close()
w.start()
w.isconnected()

Welcome to use Wind Quant API for Python (WindPy)!

COPYRIGHT (C) 2024 WIND INFORMATION CO., LTD. ALL RIGHTS RESERVED.
IN NO CIRCUMSTANCE SHALL WIND BE RESPONSIBLE FOR ANY DAMAGES OR LOSSES CAUSED BY USING WIND QUANT API FOR Python.


True

In [16]:
with open("INDEX_START.json", "r", encoding="utf-8") as f:
    INDEX_START = json.load(f)

In [17]:
df = pd.read_csv("A股指数.csv")
a_share_dict = dict(zip(df["Ticker"], df["Name"]))

df = pd.read_csv("海外指数.csv")
other_market_dict = dict(zip(df["Ticker"], df["Name"]))

print(len(a_share_dict), len(other_market_dict))

332 57


In [18]:
len(INDEX_START)

203

# Load Local Database

In [27]:
INDEX_DATA = {}

for ticker in INDEX_START:
    path = f"Data/{ticker}.csv"
    
    if os.path.isfile(path): 
        data = pd.read_csv(path, index_col = 0, parse_dates = True)
        data.index = pd.to_datetime(data.index, format="%Y-%m-%d", errors='coerce')
        data.index = data.index.date
        INDEX_DATA[ticker] = data.copy(deep = True)
        
len(INDEX_DATA)

186

In [28]:
print(INDEX_DATA['000300.SH'].index[-1])

2025-09-24


# Update Existing Tickers

In [29]:
today = "2025-09-25"
today_date = datetime.datetime.strptime(today, "%Y-%m-%d").date()
today_date

datetime.date(2025, 9, 25)

In [30]:
test_start = "2025-08-30"
temp = w.wsd('HSI.HI', 'pe_ttm,dividendyield2,close', test_start, today, '', usedf = True)[1]
temp.tail()

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-09-19,12.0401,2.9509,26545.10
2025-09-22,11.9248,2.9940,26344.14
2025-09-23,11.8520,3.0139,26159.12
2025-09-24,11.9948,2.9588,26518.65
2025-09-25,11.9505,2.9615,26484.68


In [31]:
# for ticker in tqdm(INDEX_DATA):
#     data = INDEX_DATA[ticker]

#     idx = -1
#     last = date.index[idx]
#     while not (type(last) is datetime.date):
#         idx -= 1
#         last = date.index[idx]

#     start = (last + pd.Timedelta(days = 1)).strftime("%Y-%m-%d")
    
#     new_data = w.wsd(ticker, 'pe_ttm,dividendyield2,close', start, today, '', usedf = True)[1]

#     if len(new_data) == 1 and new_data.index[0] == ticker:
#         new_data.index = [today_date]
    
#     if new_data.columns[0] == 'OUTMESSAGE':
#         print(ticker)
#         continue
    
#     INDEX_DATA[ticker] = pd.concat([data, new_data])
#     INDEX_DATA[ticker].dropna(inplace = True)

In [32]:
for ticker in tqdm(INDEX_DATA):
    # print("Processing:", ticker)
    data = INDEX_DATA[ticker]

    if data.index[-1] == today_date:
        continue
    
    start = data.index[-1].strftime("%Y-%m-%d")

    new_data = w.wsd(ticker, 'pe_ttm,dividendyield2,close', start, today, '', usedf = True)[1]

    data = pd.concat([data, new_data]).groupby(level = 0).last()
    data.index = pd.to_datetime(data.index, format="%Y-%m-%d", errors='coerce')
    data.index = data.index.date

    data = data[~data.index.duplicated(keep = "last")]

    # if len(new_data) == 1 and new_data.index[0] == ticker:
    #     new_data.index = [today_date]
    
    if new_data.columns[0] == 'OUTMESSAGE':
        print(ticker)
        continue

    INDEX_DATA[ticker] = data.copy(deep = True)
    INDEX_DATA[ticker].dropna(inplace = True)

  0%|                                                                                          | 0/186 [00:00<?, ?it/s]

Processing: NDX.GI


  1%|▍                                                                                 | 1/186 [00:00<01:36,  1.92it/s]

Processing: SPX.GI


  1%|▉                                                                                 | 2/186 [00:01<01:36,  1.91it/s]

...

 99%|███████████████████████████████████████████████████████████████████████████████▏| 184/186 [01:40<00:01,  1.87it/s]

Processing: HSISC.HI


 99%|███████████████████████████████████████████████████████████████████████████████▌| 185/186 [01:41<00:00,  1.87it/s]

Processing: TPX.GI


100%|████████████████████████████████████████████████████████████████████████████████| 186/186 [01:41<00:00,  1.83it/s]


In [35]:
data.tail()

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-09-19,18.3727,2.5740,3147.68
2025-09-22,18.4506,2.5591,3163.17
2025-09-23,18.4506,2.5591,3163.17
2025-09-24,18.4869,2.5541,3170.45
2025-09-25,18.5676,2.5430,3185.35


# Download New Tickers

In [11]:
for ticker, start in tqdm(INDEX_START.items()):
    if ticker in INDEX_DATA:
        continue
        
    assert ticker not in INDEX_DATA
    
    data = w.wsd(ticker, 'pe_ttm,dividendyield2,close', start, today, '', usedf = True)[1]
    
    junk = data['PE_TTM']

    if data.isna().sum().sum() != 0:
        print(ticker)
    
    data.dropna(inplace = True)
    
    INDEX_DATA[ticker] = data.copy(deep = True)

 99%|███████████████████████████████████████▌| 187/189 [00:00<00:00, 270.33it/s]

SP6CSSUP.SPI


100%|█████████████████████████████████████████| 189/189 [00:02<00:00, 84.13it/s]


In [36]:
for ticker, data in INDEX_DATA.items():
    path = os.path.join("Data", f"{ticker}.csv")
    data.to_csv(path, index = True, encoding = "utf-8")

In [16]:
len(INDEX_DATA)

186

In [98]:
INDEX_DATA['931151.CSI'].tail()

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-04-23,153.848694,2.4316,1928.0254
2025-04-24,147.197403,2.4611,1905.0281
2025-04-25,333.884491,2.4397,1922.7190
2025-04-28,498.459991,2.4194,1922.8445
2025-04-29,1008.526611,2.4382,1911.9818


In [20]:
temp = w.wsd("NVDA.O", 'close,volume,pe_ttm,mkt_cap_ard', start, today, '', usedf = True)[1]

In [21]:
temp.tail()

,CLOSE,VOLUME,PE_TTM,MKT_CAP_ARD
2025-08-14,182.02,129553959.0,57.848854,4.441288e+12
2025-08-15,180.45,156602161.0,57.349884,4.402980e+12
2025-08-18,182.01,132007959.0,57.845676,4.441044e+12
2025-08-19,175.64,185229219.0,55.821190,4.285616e+12
2025-08-20,175.64,NaN,55.821190,4.285616e+12


In [82]:
temp = INDEX_DATA['000300.SH']

temp.tail(30)

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-06-20,12.8262,3.2117,3846.6443
2025-06-23,12.9015,3.1666,3857.9016
2025-06-24,13.0251,3.1389,3904.0342
2025-06-25,13.1709,3.2288,3960.0662
2025-06-26,13.1643,3.3321,3946.0195
2025-06-27,13.0224,3.3039,3921.7578
2025-06-30,13.0659,3.2316,3936.0791
2025-07-01,13.1154,3.2264,3942.7620
2025-07-02,13.1225,3.2248,3943.6849
2025-07-03,13.1583,3.2269,3968.0676
